# PyTorch Deep Dive

**Course:** ML, Deep Learning & Computer Vision — Phase 3  
**Companion slides:** `Phase_03_PyTorch_Slides.pptx`  
**Prerequisites:** Phase 3 theory notebook (neurons, backprop, loss, optimisers)  

---

This notebook teaches PyTorch from the ground up. Every concept maps directly to what you already know from NumPy:

| NumPy | PyTorch | What changes |
|-------|---------|-------------|
| `np.array` | `torch.tensor` | Runs on GPU, tracks gradients |
| Manual backprop | `loss.backward()` | Automatic differentiation |
| `for` loop training | `nn.Module` + `DataLoader` | Structured, scalable |
| `np.save` | `torch.save` | Save/load model weights |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device:    {device}")

---
# 1. Tensors — GPU-ready arrays with gradients

A PyTorch tensor is a NumPy array that:
- Can live on GPU for massive parallelism
- Tracks operations for automatic differentiation

## 1.1 Creating tensors

In [ ]:
# From Python lists
a = torch.tensor([1, 2, 3, 4])
print(f"a = {a}, dtype={a.dtype}, shape={a.shape}")

# From NumPy (shares memory — no copy!)
np_arr = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(np_arr)
print(f"\nFrom numpy: {t}, dtype={t.dtype}")

# Specify dtype (float32 is the default for DL)
t32 = torch.tensor([1.0, 2.0], dtype=torch.float32)
print(f"float32: {t32}")

# Common constructors (same as NumPy!)
print(f"\nzeros:   {torch.zeros(2, 3)}")
print(f"ones:    {torch.ones(2, 3)}")
print(f"randn:   {torch.randn(2, 3)}")
print(f"arange:  {torch.arange(0, 10, 2)}")
print(f"eye:     {torch.eye(3)}")

## 1.2 Operations — same as NumPy

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print("Element-wise:", a + b, a * b, a ** 2)
print("Dot product: ", torch.dot(a, b))

# Matrix operations
X = torch.randn(3, 4)  # (3, 4)
W = torch.randn(4, 2)  # (4, 2)
output = X @ W          # (3, 2) — same @ operator as NumPy!
print(f"\nX @ W shape: {output.shape}")

# Reshaping
t = torch.arange(12)
print(f"\nreshape(3,4):  {t.reshape(3, 4).shape}")
print(f"view(2,6):     {t.view(2, 6).shape}")      # view = reshape (must be contiguous)
print(f"unsqueeze(0):  {t.unsqueeze(0).shape}")     # add dimension: (12,) → (1, 12)
print(f"squeeze:       {torch.zeros(1,3,1).squeeze().shape}")  # remove size-1 dims

## 1.3 GPU — moving tensors

`.to(device)` moves data between CPU and GPU. **Both model and data must be on the same device.**

In [ ]:
# Move to GPU (if available)
t = torch.randn(3, 3)
print(f"Before: device={t.device}")

t_gpu = t.to(device)
print(f"After:  device={t_gpu.device}")

# Operations between devices FAIL
# t + t_gpu  # RuntimeError if on different devices!

# Back to CPU (needed for NumPy, plotting)
t_cpu = t_gpu.cpu()
t_numpy = t_gpu.cpu().numpy()  # convert to NumPy (must be on CPU first)
print(f"Back to numpy: type={type(t_numpy)}")

---
# 2. Autograd — automatic differentiation

This is PyTorch's killer feature. Instead of computing gradients by hand (like we did in the theory notebook), PyTorch **records all operations** on tensors and computes gradients automatically.

## 2.1 The basics: `requires_grad=True`

In [ ]:
# Create a tensor that tracks gradients
x = torch.tensor(3.0, requires_grad=True)

# Do some computation
y = x ** 2 + 2 * x + 1  # y = x² + 2x + 1
print(f"x = {x.item():.1f}")
print(f"y = x² + 2x + 1 = {y.item():.1f}")

# Compute dy/dx automatically!
y.backward()

print(f"dy/dx = 2x + 2 = {x.grad.item():.1f}")  # at x=3: 2*3+2 = 8
print(f"\nPyTorch computed this AUTOMATICALLY by recording the computation graph.")

In [ ]:
# Multi-variable example: f(x, y) = x²y + y³
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

f = x**2 * y + y**3
f.backward()

print(f"f(x,y) = x²y + y³ = {f.item():.1f}")
print(f"∂f/∂x = 2xy = {x.grad.item():.1f}")        # 2*2*3 = 12
print(f"∂f/∂y = x² + 3y² = {y.grad.item():.1f}")   # 4 + 27 = 31
print(f"\nAutograd handles the chain rule across ANY computation — no matter how complex.")

## 2.2 Autograd with vectors — how loss.backward() works

In [ ]:
# This is what happens inside a real training step:
# 1. Forward pass (computation graph is built)
# 2. loss.backward() computes all gradients
# 3. optimizer.step() updates weights

# Simulate a tiny linear model: y = wx + b
torch.manual_seed(42)

# "Learned" parameters
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

# Data
x_data = torch.tensor([1.0, 2.0, 3.0, 4.0])
y_true = torch.tensor([2.0, 4.0, 6.0, 8.0])  # true relationship: y = 2x

# Forward
y_pred = w * x_data + b
loss = ((y_pred - y_true) ** 2).mean()  # MSE

print(f"Predictions: {y_pred.data.tolist()}")
print(f"Loss: {loss.item():.4f}")

# Backward — computes dL/dw and dL/db
loss.backward()

print(f"\ndL/dw = {w.grad.item():.4f}")
print(f"dL/db = {b.grad.item():.4f}")
print(f"\nTo reduce loss: w should increase (grad is negative = move right)")
print(f"                b stays near 0 (grad is small)")

## 2.3 Training loop — the pattern you'll use 1000 times

Every PyTorch training loop follows this exact structure:

In [ ]:
# Linear regression from scratch with autograd
torch.manual_seed(42)

# Generate data: y = 2x + 1 + noise
X = torch.linspace(-3, 3, 100).unsqueeze(1)  # (100, 1)
y = 2 * X + 1 + torch.randn(100, 1) * 0.5

# Parameters
w = torch.randn(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.01

losses = []
for epoch in range(200):
    # === 1. FORWARD PASS ===
    y_pred = X * w + b
    loss = ((y_pred - y) ** 2).mean()
    losses.append(loss.item())
    
    # === 2. BACKWARD PASS ===
    loss.backward()  # computes gradients
    
    # === 3. UPDATE WEIGHTS ===
    with torch.no_grad():  # don't track this operation
        w -= lr * w.grad
        b -= lr * b.grad
    
    # === 4. ZERO GRADIENTS ===
    w.grad.zero_()  # CRITICAL: gradients accumulate by default!
    b.grad.zero_()

print(f"Learned: y = {w.item():.3f}x + {b.item():.3f}")
print(f"True:    y = 2.000x + 1.000")
print(f"Final loss: {losses[-1]:.4f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X.numpy(), y.numpy(), s=10, alpha=0.5, color='#378ADD')
axes[0].plot(X.numpy(), (X * w + b).detach().numpy(), color='#D85A30', linewidth=2, label=f'y={w.item():.2f}x+{b.item():.2f}')
axes[0].legend(); axes[0].set_title('Fit', fontweight='bold'); axes[0].grid(True, alpha=0.2)
axes[1].plot(losses, color='#1D9E75', linewidth=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE Loss')
axes[1].set_title('Training loss', fontweight='bold'); axes[1].grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

---
# 3. `nn.Module` — building models the PyTorch way

Instead of managing weights manually, PyTorch provides `nn.Module` — a base class that:
- Holds all parameters automatically
- Moves everything to GPU with `.to(device)`
- Switches between train/eval mode (dropout, batch norm)

## 3.1 Your first nn.Module

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()  # always call this first!
        
        # Define layers (parameters are registered automatically)
        self.fc1 = nn.Linear(input_dim, hidden_dim)    # hidden layer
        self.fc2 = nn.Linear(hidden_dim, output_dim)   # output layer
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        """Define the forward pass — PyTorch builds the graph automatically."""
        x = self.fc1(x)       # linear: x @ W.T + b
        x = self.relu(x)      # activation
        x = self.dropout(x)   # dropout (only active during training)
        x = self.fc2(x)       # output layer
        return x

# Create model
model = SimpleNet(784, 256, 10)
print(model)

# Count parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")

# List all parameter tensors
print("\nParameter shapes:")
for name, param in model.named_parameters():
    print(f"  {name:15s} → {list(param.shape)}")

In [ ]:
# Forward pass
fake_batch = torch.randn(32, 784)  # 32 images, 784 pixels each
logits = model(fake_batch)          # calls model.forward() automatically
print(f"Input shape:  {fake_batch.shape}")
print(f"Output shape: {logits.shape}")  # (32, 10) — 10 class scores
print(f"\nFirst sample logits: {logits[0].detach().round(decimals=2)}")

# Apply softmax for probabilities
probs = F.softmax(logits, dim=1)
print(f"First sample probs:  {probs[0].detach().round(decimals=3)}")
print(f"Sum = {probs[0].sum().item():.4f}")

## 3.2 Using `nn.Sequential` for simple architectures

In [ ]:
# If your model is just a chain of layers, use nn.Sequential
model_seq = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 10),
)
print(model_seq)

# Works the same way
logits = model_seq(fake_batch)
print(f"\nOutput shape: {logits.shape}")

# Rule of thumb:
# - nn.Sequential for simple feed-forward chains
# - nn.Module subclass when you need conditional logic, skip connections, etc.

---
# 4. Dataset & DataLoader — feeding data efficiently

For real training, you need:
1. **Dataset** — holds your data and returns one sample at a time
2. **DataLoader** — batches, shuffles, and parallelises loading

## 4.1 Built-in TensorDataset

In [ ]:
# Simplest approach: wrap tensors in TensorDataset
X = torch.randn(1000, 784)
y = torch.randint(0, 10, (1000,))

dataset = TensorDataset(X, y)
print(f"Dataset size: {len(dataset)}")
print(f"One sample: X shape={dataset[0][0].shape}, y={dataset[0][1]}")

# DataLoader handles batching and shuffling
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Iterate
for batch_X, batch_y in loader:
    print(f"Batch: X={batch_X.shape}, y={batch_y.shape}")
    break  # just show the first batch

## 4.2 Custom Dataset

In [ ]:
class SyntheticDataset(Dataset):
    """Custom dataset — the pattern for ANY data source."""
    
    def __init__(self, n_samples=1000, n_features=20, n_classes=5):
        # Generate synthetic data
        self.X = torch.randn(n_samples, n_features)
        self.y = torch.randint(0, n_classes, (n_samples,))
    
    def __len__(self):
        """How many samples total."""
        return len(self.X)
    
    def __getitem__(self, idx):
        """Return one sample (features, label)."""
        return self.X[idx], self.y[idx]

# Usage
train_dataset = SyntheticDataset(1000)
val_dataset = SyntheticDataset(200)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} samples, {len(val_loader)} batches")

---
# 5. The complete training loop

This is the **most important pattern in PyTorch**. Every project uses this structure.

```python
for epoch in range(n_epochs):
    model.train()                    # enable dropout, batch norm training mode
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()        # 1. zero old gradients
        logits = model(batch_X)      # 2. forward pass
        loss = criterion(logits, batch_y)  # 3. compute loss
        loss.backward()              # 4. backward pass (compute gradients)
        optimizer.step()             # 5. update weights
```

Let's implement this fully, with validation, metrics, and logging.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch. Returns average loss and accuracy."""
    model.train()  # enable dropout, batch norm
    total_loss = 0
    correct = 0
    total = 0
    
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        
        optimizer.zero_grad()          # 1. zero gradients
        logits = model(X)              # 2. forward
        loss = criterion(logits, y)    # 3. loss
        loss.backward()                # 4. backward
        optimizer.step()               # 5. update
        
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        total += len(y)
    
    return total_loss / total, correct / total


@torch.no_grad()  # disable gradient tracking for validation
def evaluate(model, loader, criterion, device):
    """Evaluate on validation set. Returns average loss and accuracy."""
    model.eval()  # disable dropout, use running stats for batch norm
    total_loss = 0
    correct = 0
    total = 0
    
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = criterion(logits, y)
        
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        total += len(y)
    
    return total_loss / total, correct / total

print("Training and evaluation functions defined.")
print("Key differences:")
print("  train: model.train(), optimizer.zero_grad/step, loss.backward()")
print("  eval:  model.eval(), @torch.no_grad(), no optimizer")

In [ ]:
# === FULL TRAINING RUN ===
torch.manual_seed(42)

# Model
model = nn.Sequential(
    nn.Linear(20, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 5),
).to(device)

# Loss and optimiser
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Data
train_loader = DataLoader(SyntheticDataset(2000, 20, 5), batch_size=64, shuffle=True)
val_loader = DataLoader(SyntheticDataset(400, 20, 5), batch_size=64, shuffle=False)

# Training loop
n_epochs = 30
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(n_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{n_epochs} | "
              f"Train loss: {train_loss:.4f}, acc: {train_acc:.1%} | "
              f"Val loss: {val_loss:.4f}, acc: {val_acc:.1%}")

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, n_epochs + 1)

axes[0].plot(epochs, history['train_loss'], label='Train', color='#378ADD', linewidth=2)
axes[0].plot(epochs, history['val_loss'], label='Val', color='#D85A30', linewidth=2, linestyle='--')
axes[0].set_title('Loss', fontweight='bold'); axes[0].legend(); axes[0].grid(True, alpha=0.2)

axes[1].plot(epochs, history['train_acc'], label='Train', color='#378ADD', linewidth=2)
axes[1].plot(epochs, history['val_acc'], label='Val', color='#D85A30', linewidth=2, linestyle='--')
axes[1].set_title('Accuracy', fontweight='bold'); axes[1].legend(); axes[1].grid(True, alpha=0.2)

plt.suptitle('Training curves', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

---
# 6. Saving & loading models

Two approaches:
1. **Save state dict** (recommended) — just the weights, flexible
2. **Save entire model** — includes architecture, less portable

In [ ]:
# === Save state dict (recommended) ===
torch.save(model.state_dict(), 'model_weights.pth')
print("Saved model weights.")

# Load into a new model (must define architecture first)
new_model = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 5),
)
new_model.load_state_dict(torch.load('model_weights.pth', weights_only=True))
new_model.eval()
print("Loaded weights into new model.")

# Save full checkpoint (weights + optimizer state + epoch — for resuming training)
checkpoint = {
    'epoch': n_epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': history['train_loss'][-1],
}
torch.save(checkpoint, 'checkpoint.pth')
print("Saved full checkpoint.")

# Clean up
import os
os.remove('model_weights.pth')
os.remove('checkpoint.pth')

---
# 7. Common patterns & gotchas

### Things that trip up beginners

In [ ]:
# 1. FORGETTING to zero gradients → gradients accumulate!
print("=== Gotcha 1: gradient accumulation ===")
w = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    loss = (w * 2) ** 2
    loss.backward()
    print(f"  Step {i}: grad = {w.grad.item():.1f} (accumulates without zero_grad!)")

# 2. FORGETTING model.eval() → dropout still active during inference!
print("\n=== Gotcha 2: train vs eval mode ===")
m = nn.Dropout(0.5)
t = torch.ones(10)
m.train()
print(f"  Train mode: {m(t)}  (some zeros from dropout)")
m.eval()
print(f"  Eval mode:  {m(t)}  (all ones — dropout disabled)")

# 3. FORGETTING to move data to GPU
print(f"\n=== Gotcha 3: device mismatch ===")
print(f"  Always do: X, y = X.to(device), y.to(device)")
print(f"  Model too: model = model.to(device)")

# 4. Using .item() to get Python scalar from single-element tensor
print(f"\n=== Gotcha 4: .item() for scalars ===")
loss = torch.tensor(0.5)
print(f"  loss = {loss}  (tensor)")
print(f"  loss.item() = {loss.item()}  (Python float — use this for logging)")

---
# Summary — PyTorch cheat sheet

```python
# === SETUP ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MyModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# === TRAINING ===
model.train()
for X, y in train_loader:
    X, y = X.to(device), y.to(device)
    optimizer.zero_grad()
    loss = criterion(model(X), y)
    loss.backward()
    optimizer.step()

# === EVALUATION ===
model.eval()
with torch.no_grad():
    for X, y in val_loader:
        logits = model(X.to(device))
        preds = logits.argmax(1)

# === SAVE / LOAD ===
torch.save(model.state_dict(), 'model.pth')
model.load_state_dict(torch.load('model.pth'))
```

**Next:** The MNIST notebook applies everything above to a real dataset.